In [1]:
# ============================================================================
# STEP 0: SSL CERT AND ENVIRONMENT
# ============================================================================
import certifi
import os
from dotenv import load_dotenv

# Set SSL certificate for HTTPS connections
os.environ["SSL_CERT_FILE"] = certifi.where()
print("SSL_CERT_FILE set to:", os.environ["SSL_CERT_FILE"])

# Load environment variables from .env
load_dotenv()


SSL_CERT_FILE set to: D:\Data_Science\CV\Lib\site-packages\certifi\cacert.pem


True

In [2]:
pip install sendgrid


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Importing
import os #For env
import re
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from datetime import datetime, time, date  
import pandas as pd

In [4]:
# Init the model
llm_for_reasoning = ChatGroq(api_key = os.getenv("GROQ_API_KEY"),
               model_name = "llama-3.3-70b-versatile",
               temperature = 0)

In [5]:
import sqlite3
# Run this first, init the database
def init_sqlite_database():
    """init from init_sqlite.sql"""
    try:
        db_path = "attendance.db"

        if not os.path.exists("init_sqlitedb.sql"):
            print("No init file")
            return None

        #init because found

        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Reading and executing the file
        with open("init_sqlitedb.sql","r") as f:
            sql_script = f.read()

        cursor.executescript(sql_script)
        conn.commit() #Saving

        cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table';")
        tables = [table[0] for table in cursor.fetchall()]

        conn.close()
        print(f"Create SQLite db with tables: {tables}")
        return SQLDatabase.from_uri(f"sqlite:///{db_path}")
    except Exception as e:
        print("Failed to init sql db:")
        print(e)
        return None
def test_connection():
    try:
        db = SQLDatabase.from_uri("sqlite:///attendance.db")
        tables = db.get_usable_table_names()
        print(f"Connected, found: {tables}")
        
        result = db.run("SELECT COUNT(*) as student_count FROM students")
        print(f"📊 Students in database: {result}")
        return db
    except Exception as e:
        print(f"Connection failed: {e}")
        print("Creating a new sql db")
        return init_sqlite_database()

In [6]:
db = test_connection()

Connected, found: ['agent_analysis_log', 'attendance_sessions', 'class_schedule', 'semester_config', 'student_circumstances', 'student_daily_insights', 'students']
📊 Students in database: [(100,)]


## Helper functions

In [7]:
def parse_ai_response(ai_response, expected_values):
    """
    Parse AI response looking for lines with exactly (expected_values - 1) commas
    and exactly expected_values parts when split by commas.
    
    Args:
        ai_response (str): The raw AI response
        expected_values (int): Number of expected values (e.g., 3 for status,score,reason)
    
    Returns:
        list: Parsed values if found, None if no valid line found
    """
    try:
        expected_commas = expected_values - 1
        
        lines = ai_response.split('\n')
        for line in lines:
            line = line.strip()
            
            # Skip empty lines
            if not line:
                continue
                
            # Count commas in this line
            comma_count = line.count(',')
            
            if comma_count == expected_commas:
                parts = line.split(',')
                
                # Check if we have exactly the right number of parts
                if len(parts) == expected_values:
                    # All parts should have content (not empty after stripping)
                    cleaned_parts = [part.strip() for part in parts]
                    if all(cleaned_parts):
                        return cleaned_parts
        
        return None
        
    except Exception as e:
        logging.error(f"Error parsing AI response: {e}")
        return None

In [8]:
from datetime import datetime, time
import logging

def get_connection(row_factory = None):
    """Connect to db"""
    conn = sqlite3.connect("attendance.db")
    if row_factory:
        conn.row_factory = row_factory
    return conn
def get_current_session_direct():
    """
    Get the current session, next one if it's break time
    """
    conn = get_connection()
    try:
        cursor = conn.cursor()
        query = """
            SELECT session_number, start_time, end_time
            FROM class_schedule
            WHERE start_time<=TIME('now','localtime') AND TIME('now','localtime') <=end_time
            LIMIT 1
        """
        cursor.execute(query)
        result = cursor.fetchone()
        if result is None:
            query = """
                SELECT session_number, start_time, end_time
                FROM class_schedule
                WHERE TIME('now','localtime') < start_time
                ORDER BY session_number ASC
                LIMIT 1
            """
            cursor.execute(query)
            result = cursor.fetchone()
        if result:
            # Convert string times to time
            session_number, start_str, end_str = result
            start_time = datetime.strptime(start_str, '%H:%M:%S').time()
            end_time = datetime.strptime(end_str, '%H:%M:%S').time()

            return {
                'session_number':session_number,
                'start_time':start_time,
                'end_time':end_time
            }
        return "No active session"
    finally:
        conn.close()
def get_current_session_with_time(input_time):
    """
    Get the current session based on a specific input time
    """
    conn = get_connection()
    try:
        cursor = conn.cursor()
        
        time_str = input_time.strftime('%H:%M:%S')
        
        query = """
            SELECT session_number, start_time, end_time
            FROM class_schedule
            WHERE start_time <= ? AND ? <= end_time
            LIMIT 1
        """
        cursor.execute(query, (time_str, time_str))
        result = cursor.fetchone()
        
        if result is None:
            query = """
                SELECT session_number, start_time, end_time
                FROM class_schedule
                WHERE ? < start_time
                ORDER BY session_number ASC
                LIMIT 1
            """
            cursor.execute(query, (time_str,))
            result = cursor.fetchone()
            
        if result:
            session_number, start_str, end_str = result
            start_time = datetime.strptime(start_str, '%H:%M:%S').time()
            end_time = datetime.strptime(end_str, '%H:%M:%S').time()

            return {
                'session_number': session_number,
                'start_time': start_time,
                'end_time': end_time
            }
        return "No active session"
    finally:
        conn.close()

In [9]:
def get_session_by_number(session_number:int):
    """Get the session by nunmber"""
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute("""
        SELECT session_number, start_time, end_time
        FROM class_schedule
        WHERE session_number = ?
        """,(session_number,))

        result = cursor.fetchone()
        if result:
            session_num, start_str, end_str = result
            start_time = datetime.strptime(start_str, '%H:%M:%S').time()
            end_time = datetime.strptime(end_str, '%H:%M:%S').time()
            return {
                'session_number':session_num,
                'start_time':start_time,
                'end_time':end_time
            }
        return None
    finally:
        conn.close()

In [10]:
def calculate_late_minutes (entry_time, scheduled_start_time):
    """ calculate late minutes"""
    if isinstance(entry_time, datetime):
        entry_time = entry_time.time()
    if isinstance(scheduled_start_time, str):
        scheduled_start_time = datetime.strptime(scheduled_start_time, '%H:%M:%S').time()

    entry_datetime = datetime.combine(datetime.today(), entry_time)
    scheduled_datetime = datetime.combine(datetime.today(), scheduled_start_time)

    if entry_datetime > scheduled_datetime:
        delta = entry_datetime - scheduled_datetime
        return int(delta.total_seconds()/60)
    return 0


In [11]:
def get_student_name(student_id):
    """Get student name by ID"""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM students WHERE id = ?", (student_id,))
        result = cursor.fetchone()
        return result[0] if result else "Unknown"
def get_or_create_student(name):
    """Get or create student """
    try:
        conn = get_connection()
        cursor = conn.cursor()
        
        cursor.execute("SELECT id FROM students WHERE name = ?", (name,))
        result = cursor.fetchone()

        if not result:
            cursor.execute("INSERT INTO students (name) VALUES (?)", (name,))
            student_id = cursor.lastrowid
            conn.commit()
            logging.info(f"Created new student: {name} (ID: {student_id})")
        else:
            student_id = result[0]
            logging.info(f"Found existing student: {name} (ID: {student_id})")

        conn.close()
        return student_id
        
    except Exception as e:
        logging.error(f"Error getting/creating student: {e}")
        return None


In [12]:
def get_student_attendance_history(student_id):
    """Get the student history"""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            cursor.execute("""
                SELECT 
                    COUNT(*) as total_sessions,
                    SUM(CASE WHEN attendance_status = 'late' THEN 1 ELSE 0 END) as late_count,
                    SUM(CASE WHEN attendance_status = 'very_late' THEN 1 ELSE 0 END) as very_late_count,
                    AVG(late_minutes) as avg_late_minutes
                FROM attendance_sessions 
                WHERE student_id = ? 
                AND session_date >= DATE('now', '-7 days')
            """, (student_id,))
            result = cursor.fetchone()
            if result:
                total,late,very_late, avg_late = result
                # Cho cac phan tu bang None trong SQLITE
                late = late or 0
                very_late = very_late or 0
                avg_late = avg_late or 0

                return f"Last 7 days: {total} sessions, {late} late, {very_late} very late, avg{avg_late:.1f} min late"
            return "No recent history"
    except Exception as e:
        logging.error(f"Error while getting student hisotry:")
        return "History unavailable"

def active_check(student_id):
    """If the date is valid, active, if not, deactive"""
    try:
        
        with get_connection() as conn:
            #deactive
            cursor = conn.cursor()
            cursor.execute("""
                SELECT id, start_date, end_date
                FROM student_circumstances 
                WHERE student_id = ? AND is_active = 1
            """, (student_id,))
            results_active = cursor.fetchall()
            cursor.execute("""
                SELECT id, start_date, end_date
                FROM student_circumstances 
                WHERE student_id = ? AND is_active = 0
            """, (student_id,))
            results_deactive = cursor.fetchall()
            for result in results_active:
                cir_id , start_date, end_date = result
                start_date = datetime.strptime(start_date, "%Y-%m-%d")
                end_date = datetime.strptime(end_date, "%Y-%m-%d")
                current_date = datetime.now()
                if current_date>end_date or current_date<start_date:
                    cursor.execute("""
                        UPDATE student_circumstances
                        SET is_active = 0
                        WHERE id = ?
                    """,(cir_id,))
            #active
            for result in results_deactive:
                cir_id , start_date, end_date = result
                start_date = datetime.strptime(start_date, "%Y-%m-%d")
                end_date = datetime.strptime(end_date, "%Y-%m-%d")
                current_date = datetime.now()
                if current_date<=end_date and current_date>=start_date:
                    cursor.execute("""
                        UPDATE student_circumstances
                        SET is_active = 1
                        WHERE id = ?
                    """,(cir_id,))       
    except Exception as e:
        print(e)



def get_student_circumstances(student_id, session_number=None):
    """Get student circumstances with session-specific excuses"""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            active_check(student_id)
            if session_number:
                # Get circumstances specific to this session
                cursor.execute("""
                    SELECT circumstance_type, description, session_numbers, excuse_type
                    FROM student_circumstances 
                    WHERE student_id = ? AND is_active = 1
                    AND date('now') BETWEEN start_date AND end_date
                    AND (session_numbers = 'all' OR session_numbers LIKE ?)
                """, (student_id, f'%{session_number}%'))
            else:
                # Get all active circumstances
                cursor.execute("""
                    SELECT circumstance_type, description, session_numbers, excuse_type
                    FROM student_circumstances 
                    WHERE student_id = ? AND is_active = 1
                    AND date('now') BETWEEN start_date AND end_date
                """, (student_id,))

            results = cursor.fetchall()
            if results:
                circumstances = []
                for circ_type, description, session_nums, excuse_type in results:
                    if session_nums and excuse_type:
                        circumstances.append(f"{circ_type}({excuse_type} for sessions:{session_nums}):{description}")
                    else:
                        circumstances.append(f"{circ_type}:{description}")
                return " | ".join(circumstances)
            return "No active circumstances"
    except Exception as e:
        logging.error(f"Error getting student circumstances: {e}")
        return "Circumstances unavailable"

In [13]:
def calculate_auto_fill_score(student_id, session_num, llm, student_name, is_entry):
    """Determining the score based on circumstances and stuff"""

    student_circumstances = get_student_circumstances(student_id, session_num)
    if is_entry:
        # Check circumstances for sessions BEFORE current
        ai_prompt = f"""
        AUTO-FILL SCORING FOR ENTRY:
        
        STUDENT: {student_name}
        SESSION: {session_num} (session being auto-filled)
        FIRST ENTRY: TRUE
        CIRCUMSTANCES: {student_circumstances}
        
        SCORING RULES FOR SESSIONS BEFORE ENTRY (STRICT - FOLLOW EXACTLY):
        - 'excused' (score: 1.0): Student has 'full' excuse for this specific session
        - 'excused' (score: 0.5): Student has 'partial' or 'late_arrival' excuse for this session  
        - 'absent' (score: 0.0): No valid documented excuse for this session
        
        STATUS RULES FOR FIRST ENTRY:
        • Use 'excused' ONLY if circumstances specifically mention this session number
        • Use 'absent' if no valid excuse exists
        • DO NOT use 'on_time' or 'late' for auto-filled first entry sessions
        
        Return ONLY: status,score,reason
        Valid Examples:
        excused,1.0,has_medical_excuse_for_session_{session_num}
        excused,0.5,has_transportation_issues_documented
        absent,0.0,no_documented_excuse_for_this_session
        
        Your decision (ONLY use 'excused' or 'absent'):
        """
    else:
        # Between last entry and current exit
        ai_prompt = f"""
        AUTO-FILL SCORING FOR MISSED SESSION:
        
        STUDENT: {student_name}
        SESSION: {session_num} (session being auto-filled)
        CIRCUMSTANCES: {student_circumstances}
        
        SCORING RULES FOR MISSED SESSIONS BETWEEN RECORDED ENTRIES (STRICT - FOLLOW EXACTLY):
        - 'on_time' (score: 1.0): Assume student was present but forgot to record entry (default)
        - 'absent' (score: 0.0): Only if clear evidence of absence from circumstances
        
        STATUS RULES FOR MISSED SESSIONS:
        • Use 'on_time' as default assumption (student was present between scans)
        • Use 'absent' ONLY if clear evidence they were missing
        • DO NOT use 'late' or 'excused' for auto-filled between sessions
        
        Return ONLY: status,score,reason
        Valid Examples:
        on_time,1.0,assumed_present_between_recorded_sessions
        on_time,1.0,student_likely_present_based_on_movement_pattern
        absent,0.0,clear_evidence_of_absence_from_circumstances
        
        Your decision (ONLY use 'on_time' or 'absent'):
        """
    
    ai_response = llm.invoke(ai_prompt).content.strip()
    print(f"AI Response: {ai_response}")
    
    parsed_values = parse_ai_response(ai_response, 3)
    
    if parsed_values and len(parsed_values) == 3:
        status, score_str, reason = parsed_values
        try:
            score = float(score_str)
            
            if is_entry:
                allowed_statuses = ['excused', 'absent']
            else:
                allowed_statuses  = ['on_time', 'absent']
            
            if status not in allowed_statuses:
                logging.warning(f"Invalid status '{status}' for context, using fallback")
                status = 'absent' if is_entry else 'on_time'
                score = 0.0 if is_entry else 1.0
                reason = f'fallback_invalid_status_{reason}'
            
            # Validate score range
            if score < 0 or score > 1:
                score = max(0.0, min(1.0, score))
                
            return {
                'status': status,
                'score': score,
                'reason': reason
            }
            
        except ValueError:
            logging.warning(f"Invalid score format: {score_str}")
    
    # Fallback if parsing fails - ALWAYS return a dictionary
    logging.warning(f"Could not parse AI auto-fill response, using default")
    fallback_status = 'absent' if is_entry else 'on_time'
    fallback_score = 0.0 if is_entry else 1.0
    
    return {
        'status': fallback_status,
        'score': fallback_score,
        'reason': 'auto_fill_parse_error_using_default'
    }

In [14]:
def auto_fill_missing_sessions(student_id,last_session_num, current_session_num, llm, student_name, is_entry):
    """ Auto fill the session before the first entry
    and the session between the last entry and the nearest exist"""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            filled_sessions = []
            current_date = datetime.now().strftime('%Y-%m-%d')
            # #Get the last session_num from database
            # cursor.execute("""
            #     SELECT MAX(CAST(session_number AS INTEGER)) 
            #     FROM attendance_sessions 
            #     WHERE student_id = ? AND session_date = ?
            # """, (student_id, current_date))
            # result = cursor.fetchone()
            #If none session, default to 0
            # last_session_num = result[0] if result[0] else 0
            print(f"Last session num was: {last_session_num}")
            print(f"current session num was: {current_session_num}")
            
            for session_num in range(last_session_num + 1, current_session_num):
                session_info = get_session_by_number(session_num)
                print(f"Checking session {session_num}, filled_sessions: {filled_sessions}")
                if session_info:
                    auto_fill_result = calculate_auto_fill_score(student_id, session_num, llm, student_name, is_entry)
                    
                    # Use the dictionary values correctly
                    status = auto_fill_result['status']
                    score = auto_fill_result['score']
                    reason = auto_fill_result['reason']
                    

                    cursor.execute("""
                        INSERT INTO attendance_sessions 
                        (student_id, session_date, entry_time, status, attendance_status, 
                         session_number, reason_for_scoring,attendance_score, late_minutes)
                        VALUES (?, ?, ?, ?, ?, ?, 
                               ?,?, 0)
                    """, (
                        student_id, current_date, 
                        datetime.now().strftime('%H:%M:%S'),
                        "present" if status!="absent" and is_entry==True else "left",
                        status,
                        session_num,
                        f"AUTO_FILLED: {reason} (score:{score})",
                        score
                    ))                    
                    filled_sessions.append({
                        'session': session_num,
                        'status': status,
                        'score': score,
                        'reason': reason
                    })

                    logging.info(f"Auto-filled session {session_num} for {student_name}, ID: {student_id} with status '{status}' and score {score}")
            
            conn.commit()
            return filled_sessions

    except Exception as e:
        logging.error(f"Auto fill score failed: {e}")
        return []

In [15]:
def record_entry(name, llm , active_session = None):
    """Recording Entry. Note: the none values are just for debugging"""
    try:
        student_id = get_or_create_student(name)
        if student_id is None:
            return None
        with get_connection() as conn:
            cursor = conn.cursor()
            current_datetime = datetime.now()
            current_date = current_datetime.strftime('%Y-%m-%d')
            current_time = current_datetime.time()
            session_info = get_current_session_direct()
            # Find the last exit_time
            cursor.execute("""
                SELECT exit_time
                FROM attendance_sessions
                WHERE student_id = ? AND session_date = ? AND exit_time IS NOT NULL
                ORDER BY exit_time DESC
                LIMIT 1
            """, (student_id, current_date))
            
            result = cursor.fetchone()
            if result and result[0] is not None:
                # Convert string to time object
                last_exit_time_str = result[0]
                last_exit_time = datetime.strptime(last_exit_time_str, '%H:%M:%S').time()
            else:
                last_exit_time = None

            
            if active_session is not None:
                if isinstance(active_session, int):
                    session_info = get_session_by_number(active_session)
                else:
                    session_info = active_session
                print(f"After override: Session {session_info}") 
                
            if not session_info:
                logging.warning("No active session found")
                return None

            # Get the session number from the last exit time
            if last_exit_time is not None:
                last_session_info = get_current_session_with_time(last_exit_time)
                last_session_num = last_session_info['session_number'] if isinstance(last_session_info, dict) else 0
            else:
                last_session_num = 0
                    
            #If last lession_num is zero -> first_entry
    
            current_session_num = session_info['session_number']

            #Not allowing a second entry
            if last_session_num ==current_session_num:
                return None
            
            # Auto filling for entry
            auto_fill_missing_sessions(student_id, last_session_num, current_session_num, llm,name, True)
        

            #To rework from here!!
            
            late_minutes = calculate_late_minutes(current_time, session_info['start_time'])

            # Get circumstances for THIS specific session
            student_history = get_student_attendance_history(student_id)
            student_circumstances = get_student_circumstances(student_id, session_info['session_number'])
            
            ai_prompt = f"""
            ATTENDANCE DECISION MAKING:
            
            STUDENT PROFILE:
            - Name: {name} (ID: {student_id})
            - Current Time: {current_time}
            - Session: {session_info['session_number']} ({session_info['start_time']}-{session_info['end_time']})
            - Late by: {late_minutes} minutes
            
            HISTORICAL CONTEXT:
            {student_history}
            
            PERSONAL CIRCUMSTANCES:
            {student_circumstances}
            
            DECISION MATRIX (STRICT RULES - FOLLOW EXACTLY):
            - 'on_time' (score: 1.0): Arrived within 5 minutes of session start
            - 'late' (score: 0.1-0.9): Arrived 5-60 minutes late, adjust score based on circumstances
            - 'absent' (score: 0.0): Arrived 60+ minutes late OR no valid circumstances for extreme lateness
            - 'excused' (score: 1.0): Has valid documented excuse for this specific session
            
            EXCUSE RULES:
            • Use 'excused' ONLY if circumstances specifically mention this session number
            • 'full' excuse type = completely excused regardless of arrival time
            • 'late_arrival' excuse type = excused for being late to this session
            
            SCORING GUIDELINES FOR 'late' STATUS:
            • 0.8-0.9: 5-15 min late with valid circumstances
            • 0.6-0.7: 15-30 min late with mitigating factors  
            • 0.4-0.5: 30-45 min late with minor circumstances
            • 0.1-0.3: 45-60 min late with weak or no valid reasons
            
            Return ONLY: status,score,reason_for_scoring
            Valid Examples:
            on_time,1.0,arrived_within_5_minute_grace_period
            late,0.8,15_min_late_due_to_documented_medical_appointment
            excused,1.0,has_medical_excuse_for_session_3
            absent,0.0,75_min_late_no_valid_circumstances
            
            Your decision (ONLY use 'on_time', 'late', 'absent', or 'excused'):
            """
            ai_response = llm.invoke(ai_prompt).content.strip()
            print(ai_response)
            # Parsing and checking for error
            parsed_values = parse_ai_response(ai_response, 3)
            if not parsed_values:
                logging.warning(f"Failed to parse AI response: {ai_response}, using fallback")
                if late_minutes <= 5:
                    status, score, reason_for_scoring = 'on_time', 1.0, 'fallback_grace_period'
                elif late_minutes <= 60:
                    status, score, reason_for_scoring = 'late', max(0.1, 1.0 - (late_minutes / 60)), 'fallback_late'
                else:
                    status, score, reason_for_scoring = 'absent', 0.0, 'fallback_absent'
            else:
                status, score, reason_for_scoring = parsed_values
            #Constraints
            allowed_statuses = ['on_time', 'late', 'absent', 'excused']
            if status not in allowed_statuses:
                logging.warning(f"AI returned invalid status: {status}, defaulting to 'late'")
                status = 'late'
            
            
            attendance_status = status
            attendance_score = float(score)
            ai_reason = reason_for_scoring

            if late_minutes > 0:
                logging.warning(f"🚨 {name} is LATE by {late_minutes} minutes!")
                print(f"\n{'=' * 80}")
                print(f"🤖 ADVANCED AI ATTENDANCE DECISION ENGINE")
                print(f"{'=' * 80}")
                print(f"👤 Student: {name} (ID: {student_id})")
                print(f"⏰ Scheduled: {session_info['start_time']}")
                print(f"🕒 Arrived: {current_time.strftime('%H:%M:%S')}")
                print(f"⚠️  Late by: {late_minutes} minutes")
                print(f"📊 Historical Pattern: {student_history}")
                print(f"🎯 Personal Circumstances: {student_circumstances}")
                print(f"🤖 AI Decision: {attendance_status}")
                print(f"⭐ AI Score: {attendance_score}")
                print(f"💡 Reason for Scoring: {ai_reason}")
                print(f"{'=' * 80}\n")
            else:
                logging.info(f"✅ {name} - AI Decision: {attendance_status}")
                print(f"\n{'=' * 60}")
                print(f"✅ AI ATTENDANCE CONFIRMED")
                print(f"{'=' * 60}")
                print(f"Student: {name}")
                print(f"Status: {attendance_status}")
                print(f"Score: {attendance_score}")
                print(f"Reason: {ai_reason}")
                print(f"{'=' * 60}\n")

            # Store only time in entry_time, not full datetime
            cursor.execute("""
                INSERT INTO attendance_sessions 
                (student_id, session_date, entry_time, status, attendance_status, late_minutes, reason_for_scoring,attendance_score, session_number)
                VALUES (?, ?, ?, 'present', ?, ?, ?,?, ?)
            """, (
                student_id, 
                current_date,  # Date goes here
                current_time.strftime('%H:%M:%S'), 
                attendance_status, 
                late_minutes, 
                ai_reason,
                attendance_score,
                session_info['session_number']  # ADD session_number
            ))

            session_id = cursor.lastrowid
            conn.commit()

            logging.info(f"🤖 Advanced AI attendance recorded for {name}: {attendance_status} - {ai_reason}")
            
            return {
                'session_id': session_id,
                'student_id': student_id,
                'status': attendance_status,
                'score': attendance_score,
                'late_minutes': late_minutes,
                'reason_for_scoring': ai_reason,
                'timestamp': current_datetime,
                'circumstances_considered': student_circumstances,
                'historical_context': student_history
            }
    except Exception as e:
        print(e)
        return None
                

In [16]:
#Test
# record_entry("Emily Johnson", llm_for_reasoning, 4)

In [17]:
def record_exit(name, llm, active_session=None, early_departure_reason=None):
    """Record exit with auto-filling. Note: the none values are just for debugging"""
    try:
        student_id = get_or_create_student(name)
        if student_id is None:
            return None
        with get_connection() as conn:
            cursor = conn.cursor()
            current_date = datetime.now().strftime('%Y-%m-%d')
            current_time = datetime.now().time()
            session_info = get_current_session_direct()

            # Find the last entry time for auto filling
            cursor.execute("""
                SELECT entry_time, session_number
                FROM attendance_sessions
                WHERE student_id = ? AND session_date = ?
                ORDER BY session_number DESC, entry_time DESC
                LIMIT 1
            """, (student_id, current_date))

            result = cursor.fetchone()

            if result and result[0] is not None:
                # Convert string to time object
                last_entry_time_str = result[0]
                current_session_id = result[1]  # Get the session ID for the UPDATE
                last_entry_time = datetime.strptime(last_entry_time_str, '%H:%M:%S').time()
            else:
                last_entry_time = None
                current_session_id = None
            print(last_entry_time)
            if active_session is not None:
                if isinstance(active_session, int):
                    session_info = get_session_by_number(active_session)
                else:
                    session_info = active_session

            if not session_info:
                logging.warning("The class has already ended!!")
                return None

            # Get the session number from the last entry time
            if last_entry_time is not None:
                last_session_info = get_current_session_with_time(last_entry_time)
                last_session_num = last_session_info['session_number'] if isinstance(last_session_info, dict) else 0
            else:
                last_session_num = 0

            current_session_num = session_info["session_number"]

            # Auto filling for exit
            auto_fill_missing_sessions(student_id, last_session_num, current_session_num, llm, name, False) # Not an entry
            
            # Calculating score part
            entry_time = last_entry_time  
            current_datetime = datetime.now() 
            entry_datetime = datetime.combine(current_datetime.date(), entry_time)     
            
            # Early leaving duration
            duration_minutes = max(1, int((current_datetime - entry_datetime).total_seconds() / 60))
            session_end_time = session_info['end_time']  
            early_departure_minutes = 0
            
            if current_time < session_end_time:
                end_dt = datetime.combine(current_datetime.date(), session_end_time)
                early_departure_minutes = max(0, int((end_dt - current_datetime).total_seconds() / 60))
            
            student_history = get_student_attendance_history(student_id)
            student_circumstances = get_student_circumstances(student_id, session_info['session_number'])
            
            ai_prompt = f"""
            EARLY DEPARTURE PENALTY CALCULATION:
            
            STUDENT: {name}
            SESSION: {session_info['session_number']} ({session_info['start_time']}-{session_info['end_time']})
            EARLY DEPARTURE: {early_departure_minutes} minutes early
            REASON: {early_departure_reason or 'Not specified'}
            
            HISTORICAL PATTERNS:
            {student_history}
            
            CIRCUMSTANCES:
            {student_circumstances}
            
            PENALTY MATRIX:
            - 0-5 min early: 10% penalty (score: 0.9)
            - 6-15 min early: 30% penalty (score: 0.7)  
            - 16-30 min early: 60% penalty (score: 0.4)
            - 31+ min early: 90% penalty (score: 0.1)
            - Medical/emergency: No penalty (score: 1.0)
            - Pre-approved: Reduced penalty
            
            IMPORTANT: Respond ONLY in this exact format: final_score,penalty_reason
            - final_score: number between 0.1 and 1.0
            - penalty_reason: short_description_without_spaces
            
            Examples:
            0.7,left_12_minutes_early_30_percent_penalty
            1.0,medical_appointment_no_penalty
            0.4,left_25_minutes_early_60_percent_penalty
            0.9,left_3_minutes_early_10_percent_penalty
            
            Do NOT include any explanations, just the score and reason separated by comma.
            
            Your response:
            """
            
            ai_response = llm.invoke(ai_prompt).content.strip()

            # Parsing
            final_score, penalty_reason = parse_ai_response(ai_response, 2)
            try:
                final_score_float = float(final_score)
            except ValueError:
                final_score_float = 0.7  # Default fallback
                        
            cursor.execute("""
                UPDATE attendance_sessions
                SET exit_time = ?, 
                    duration_minutes = ?,
                    status = 'left',
                    attendance_status = 'left_early',
                    reason_for_scoring = ?,
                    attendance_score = ?
                WHERE student_id = ? AND session_date = ? AND session_number = ?
            """, (
                current_time.strftime('%H:%M:%S'), 
                duration_minutes,
                f"EARLY_EXIT: {penalty_reason} (final_score:{final_score})",
                final_score,
                student_id,
                current_date,
                current_session_num
            ))
            
            conn.commit()

            # Display penalty analysis
            print(f"\n{'=' * 70}")
            print(f"⚠️  EARLY DEPARTURE PENALTY ANALYSIS")
            print(f"{'=' * 70}")
            print(f"Student: {name}")
            print(f"Session: {session_info['session_number']}")
            print(f"Duration: {duration_minutes}/45 minutes")
            print(f"Early Departure: {early_departure_minutes} minutes")
            print(f"Final Score: {final_score}")
            print(f"Penalty Reason: {penalty_reason}")
            print(f"{'=' * 70}\n")

            logging.info(f"⚠️ Early departure penalty for {name}: {final_score} - {penalty_reason}")
            
            return {
                'student_id': student_id,
                'duration_minutes': duration_minutes,
                'early_departure_minutes': early_departure_minutes,
                'final_score': float(final_score),
                'penalty_reason': penalty_reason,
                'auto_filled_sessions': True
            }

    except Exception as e:
        logging.error(f"❌ Enhanced exit recording error: {e}")
        return None

In [18]:
#Testing
#record_exit("Emily Johnson", llm_for_reasoning, 4)

### For handling errors

In [19]:
# from langchain.tools import tool
# from langchain.tools import wrap_tool_call

# @wrap_tool_call
# def handle_tool_errors(request, handler):
#     """Handle tool execution errors with custom messages."""
#     try:
#         return handler(request)
#     except Exception as e:
#         return ToolMessage(
#             content=f"Tool error: Please check your input and try again. ({str(e)})",
#             tool_call_id=request.tool_call["id"]
#         )


### Tools

In [20]:
from langchain.tools import tool
def get_expected_students():
    """
    Get the complete list of all enrolled students with their IDs and names. 
    Returns a list of tuples like [(1, "John Smith"), (2, "Jane Doe"), ...]
    """
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            cursor.execute("""
            SELECT id, name
            FROM students
            """)
            return cursor.fetchall()
    except Error as e:
        print(e)
        return None

def get_present_students():
    """
    Get only the student IDs of students who are marked present today. 
    Returns a list of student IDs like [1, 2, 3] or "No present students".
    """
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            current_date = datetime.now().strftime('%Y-%m-%d')
            cursor.execute("""
            SELECT DISTINCT student_id
            FROM attendance_sessions
            WHERE session_date = ?
            """,(current_date,))
            result = cursor.fetchall()
            # Extract just the IDs from tuples
            return [row[0] for row in result] if result else "No present students"
    except Error as e:
        print(e)
        return None
def get_missing_students():
    """
    Get the list of students who are missing today.
    Returns a list of tuples with (id, name) for missing students.
    """
    students = get_expected_students()
    presented = get_present_students()
    
    if presented == "No present students":
        return [student[0] for student in students] # All students are missing
    

    present_ids = set(presented)
    
    return [student[0] for student in students if student[0] not in present_ids]

def notify_missing_students(missing_students):
    """
    Send notifications to missing students. 
    Input should be a list of student IDs that are absent today.
    Call this after identifying which students are missing by comparing 
    expected students with present students.
    Example: [1, 5, 7, 9] or [1, 3, 8, 12]
    """
    print(f"Notifying missing students: {missing_students}")



### Multi-tool

In [21]:
@tool
def attendance_manager(action: str):
    """
    Handle all attendance-related actions in one reliable tool.
    
    Supported actions:
    - 'get_missing': Returns list of missing student IDs
    - 'notify_missing': Finds and notifies missing students parents
    - 'get_expected': Returns all expected students
    - 'get_present': Returns present student IDs
    """
    try:
        if action == "get_expected":
            return get_expected_students()
                
        elif action == "get_present":
            return get_present_students()
                
        elif action == "get_missing":
            return get_missing_students()
            
        elif action == "notify_missing":
            # Get missing students and notify all
            missing_ids = get_missing_students()
            missing_to_notify = missing_ids
            notify_missing_students(missing_to_notify)
            return f"Successfully notified {len(missing_to_notify)} missing students: {missing_to_notify}"
            
        else:
            return f"Unknown action: {action}. Use 'get_missing', 'notify_missing', 'get_expected', or 'get_present'"
            
    except Exception as e:
        return f"Error: {str(e)}"


## Things to add

When to Use Multiple Meta Tools:
### Tool 1: Attendance Manager (your existing one)

### def attendance_manager(action: str):
    """
    Handle all CORE attendance operations.
    Actions: 'get_missing', 'notify_missing', 'get_expected', 'get_present'
    """
### Tool 2: Analytics & Reports

### def analytics_manager(action: str):
    """
    Handle attendance ANALYTICS and reporting.
    Actions: 'weekly_report', 'trends', 'at_risk_students', 'predict_attendance'
    """
    # Advanced analytics, predictions, trends
### Tool 3: Student Services
  
### def student_services_manager(action: str):
    """
    Handle student INFORMATION and services.
    Actions: 'find_student', 'contact_info', 'attendance_history', 'parent_contact'
    """
    # Student lookup, contact, history
### Tool 4: System Admin

### def system_admin_manager(action: str):
    """
    Handle SYSTEM administration tasks.
    Actions: 'backup_data', 'export_reports', 'system_status', 'cleanup'
    """
    # System maintenance, backups, exports

### 1. Basic Checks
- Check current attendance status
- List present students  
- List missing students
- Get individual student status

### 2. Notifications
- Notify missing students
- Notify specific students
- Send reminders to late students
- Contact parents of absent students

### 3. Data Management  
- Get all student records
- Update attendance manually
- Add absence excuses/notes
- Mark students as present/absent
### 4. Reports
- Generate daily summary report
- Create weekly/monthly reports
- Export attendance data
- Print class rosters

### 5. Analytics
- Calculate attendance rates
- Identify attendance trends  
- Flag at-risk students
- Show class comparisons
### 6. System Operations
- Backup attendance data
- System status check
- Data cleanup/maintenance
- User management

### 7. Integration
- Sync with gradebook
- Export to school system
- Generate compliance reports
### 8. Intelligence
- Predict future attendance
- Suggest interventions
- Auto-flag patterns
- Generate insights

agent = create_agent(
    model=llm,
    tools=[
        attendance_manager,      # Core attendance ops
        analytics_manager,       # Reports & analytics  
        attendance_recording_manager, # Record_entry and record_exists
        student_services_manager, # Student info & services
        system_admin_manager     # System maintenance
        
    ],
    middleware=[handle_tool_errors],
)

In [22]:
from langchain.agents import create_agent
llm_for_agent = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=os.getenv("GROQ_API_KEY")
)
# Should use from langchain.agents.middleware import TodoListMiddleware
agent = create_agent(
    model=llm_for_agent,
    tools=[attendance_manager],
)


In [23]:
def generate_daily_insights():
    """
    This is a tool for the agent to generate daily insights at the end of the day to be
    used for recommendation, prioritizing students, etc..
    """
    try:
        conn = get_connection()
        cursor = conn.cursor()
        today = date.today().isoformat()
        #Check all the active circumstances first!
        result = cursor.execute("""
                                SELECT student_id
                                FROM student_circumstances
                                """).fetchall()
        for student_id in result:
            active_check(student_id[0])
        
        cursor.execute("DELETE FROM student_daily_insights WHERE date = ?", (today,))
        
        cursor.execute("""
            INSERT INTO student_daily_insights 
            (date, student_id, student_name, sessions_attended, sessions_late, full_day_absent, has_circumstances, priority_score)
            SELECT 
                ? as date,
                s.id as student_id, 
                s.name as student_name,
                COUNT(CASE WHEN a.attendance_status IN ('on_time', 'late') THEN 1 END) as sessions_attended,
                COUNT(CASE WHEN a.attendance_status = 'late' THEN 1 END) as sessions_late,
                CASE WHEN COUNT(a.id) = 0 THEN 1 ELSE 0 END as full_day_absent,
                CASE WHEN sc.id IS NOT NULL THEN 1 ELSE 0 END as has_circumstances,
                CASE 
                    WHEN COUNT(a.id) = 0 THEN 10
                    WHEN COUNT(CASE WHEN a.attendance_status = 'late' THEN 1 END) > 2 THEN 8
                    WHEN COUNT(CASE WHEN a.attendance_status = 'late' THEN 1 END) > 0 THEN 5
                    ELSE 1
                END as priority_score
            FROM students s
            LEFT JOIN attendance_sessions a ON s.id = a.student_id AND a.session_date = ?
            LEFT JOIN student_circumstances sc ON s.id = sc.student_id AND sc.is_active = 1
            GROUP BY s.id, s.name
        """, (today, today))
        
        conn.commit()
        conn.close()
        return f"Insights generated for {today}"
    except Exception as e:
        print(e)
        return None

In [24]:
generate_daily_insights()

'Insights generated for 2025-12-13'

In [25]:
def get_weekly_attendance_stats(end_date=None, days_back=7):
    """Get stats for any period"""
    if end_date is None:
        end_date = datetime.today().strftime('%Y-%m-%d')
    
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT date(?, ?)", (end_date, f'-{days_back} days'))
        start_date = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT 
                date(session_date) as day,
                ROUND(COUNT(CASE WHEN attendance_status = 'on_time' OR 'excused' THEN 1 END) * 100.0 / COUNT(*), 1) as on_time_pct,
                ROUND(COUNT(CASE WHEN attendance_status = 'late' THEN 1 END) * 100.0 / COUNT(*), 1) as late_pct,
                ROUND(COUNT(CASE WHEN attendance_status = 'absent' THEN 1 END) * 100.0 / COUNT(*), 1) as absent_pct
            FROM attendance_sessions 
            WHERE session_date >= ? AND session_date <= ?
            GROUP BY date(session_date)
            ORDER BY day
        """, (start_date, end_date))
        return cursor.fetchall()

In [26]:
#Remember to change the avg score to 5 later
def get_high_priority_students():
    """ 
    Get students with average priority > 5 in last 7 days
    """
    try:
        conn = get_connection()
        cursor = conn.cursor()
        today = date.today().isoformat()
        
        cursor.execute("""
        SELECT 
            student_id,
            student_name,
            ROUND(AVG(priority_score), 2) as avg_priority,
            COUNT(*) as days_with_issues
        FROM student_daily_insights 
        WHERE date >= date('now', '-7 days') AND priority_score > 1
        GROUP BY student_id, student_name
        HAVING AVG(priority_score) >=8
        ORDER BY avg_priority DESC;
        """)
        return cursor.fetchall()
    except Exception as e:
        print(e)
        return None
    finally:
        conn.close()

In [27]:
def get_intervention_history(student_id):
    """Get recent intervention history for a student"""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT reason,recommendation, intervention_effective, analysis_date
            FROM agent_analysis_log 
            WHERE student_id = ? 
            AND intervention_effective IS NOT NULL
            ORDER BY analysis_date DESC 
            LIMIT 3  -- Get last 3 interventions for pattern recognition
        """, (student_id,))
        results = cursor.fetchall()
        return results if results else None

In [28]:
get_intervention_history(59)

[('1 absence, no valid circumstances, previous intervention None',
  'Contact parents',
  0,
  '2025-11-27'),
 ('1 absence, no valid circumstances, previous intervention None',
  'Contact parents',
  0,
  '2025-11-27'),
 ('1 absence, no valid circumstances, previous intervention None',
  'Contact parents',
  1,
  '2025-11-27')]

In [29]:
def analysis_student_problems(llm):
    """
    Analysis student problems in batches of 5 with intervention history consideration
    """
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            high_priority_students = get_high_priority_students()
            
            high_priority_students = [[student_id, a, b, c, get_student_circumstances(student_id), get_intervention_history(student_id)] for student_id,a,b,c in high_priority_students]
            
            analysis_list = []
            for i in range(0, min(6, len(high_priority_students)), 3):
                batch_students = high_priority_students[i:i+3]
                
                prompt = f"""
                Analyze these {len(batch_students)} students with circumstances AND past intervention history:
                {batch_students}
                
                Output ONLY lines in format:
                student_id|alert_level|reason|recommendation
                
                **STRICT RULES - NO HALLUCINATION:**
                - Check if circumstances list is EMPTY [] or None → NO valid circumstances
                - Check if circumstances match the attendance issue type
                - CONSIDER PAST INTERVENTION EFFECTIVENESS when making recommendations
                
                **Decision Rules:**
                - HIGH: 3+ absences AND no valid circumstances
                - MEDIUM: 1-2 absences AND no valid circumstances  
                - LOW: ONLY if circumstances list is NOT empty AND circumstances are active AND cover today
                
                **INTERVENTION HISTORY CONSIDERATION:**
                - intervention_effective = 1: Previous intervention WORKED
                - intervention_effective = 0: Previous intervention FAILED  
                - intervention_effective = NULL: Intervention not yet evaluated (too recent)
                - No records: First time intervention
                
                **SPECIFIC RECOMMENDATION STRATEGIES - BE SPECIFIC:**
                - WORKED (1): "Continue [specific successful approach]"
                - FAILED (0): CHOOSE ONE: "Escalate to counselor", "Schedule in-person meeting", "Home visit", "Academic support referral", "Behavioral intervention"
                - NULL (recent): "Follow up on [recent intervention]" 
                - NO HISTORY: "Contact parents", "Schedule meeting", "Monitor attendance"
                
                **ESCALATION PATH FOR FAILED INTERVENTIONS:**
                - 1st failure: Try different contact method (phone → in-person)
                - 2nd failure: Escalate to school counselor
                - 3rd+ failure: Involve administration/principal
                
                **VALID CIRCUMSTANCES REQUIRE:**
                - is_active=1
                - Current date between start_date and end_date  
                - excuse_type matches the issue (late_arrival for lateness, etc.)
                
                **Examples - BE SPECIFIC:**
                49|medium|1 absence, previous parent contact failed|Escalate to school counselor
                59|medium|2 absences, previous meeting ineffective|Schedule in-person parent conference
                1|low|Valid circumstances, bus monitoring worked|Continue bus schedule coordination
                
                Start with student_id. No extra text.
                """
                                                
                response = llm.invoke(prompt)
                analysis_text = response.content
                
                # Parsing the response
                for line in analysis_text.strip().split('\n'):
                    if '|' in line:
                        parts = line.split('|')
                        if len(parts) >= 4:
                            clean_id = parts[0].replace('.', '').strip()
                            analysis_list.append({
                                'student_id': int(clean_id),
                                'student_name': get_student_name(clean_id),
                                'alert_level': parts[1].strip(), 
                                'reason': parts[2].strip(),
                                'recommendation': parts[3].strip()
                            })
                
                # INSERT into database
                for student in analysis_list:
                    try:
                        cursor.execute("""INSERT INTO agent_analysis_log 
                                (student_id, student_name, alert_level, reason, recommendation, created_at, analysis_date)
                                VALUES (?, ?, ?, ?, ?, ?, ?)""",
                               (student['student_id'], student['student_name'],
                                student['alert_level'], student['reason'],
                                student['recommendation'], date.today().isoformat(),
                                date.today().isoformat()))
                    except Exception as e:
                        print(f" Failed to insert {student['student_id']}: {e}")
                        continue 
                
                conn.commit()  
            return analysis_list
    except Exception as e:
        logging.warn(f"Error in analysis_student_problems: {e}")
        return None


In [30]:
def evaluate_past_interventions():
    """
    Mark previous interventions as worked or not based on attendance rate comparison
    """
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            
            # Get unevaluated interventions from 3-10 days ago
            cursor.execute("""
                SELECT 
                    a.id as log_id,
                    a.student_id,
                    a.analysis_date
                FROM agent_analysis_log a
                WHERE a.analysis_date BETWEEN date('now', '-10 days') AND date('now', '-3 days')
                AND a.intervention_effective IS NULL
            """)
            
            interventions = cursor.fetchall()
            
            for log_id, student_id, analysis_date in interventions:
                # Get attendance rate before intervention (3 days before)
                cursor.execute("""
                    SELECT COUNT(*) FROM attendance_sessions 
                    WHERE student_id = ? 
                    AND session_date BETWEEN date(?, '-3 days') AND ?
                    AND attendance_status IN ('on_time', 'late')
                """, (student_id, analysis_date, analysis_date))
                before_count = cursor.fetchone()[0]
                
                # Get attendance rate after intervention (3 days after)  
                cursor.execute("""
                    SELECT COUNT(*) FROM attendance_sessions 
                    WHERE student_id = ? 
                    AND session_date BETWEEN date(?, '+1 day') AND date(?, '+4 days')
                    AND attendance_status IN ('on_time', 'late')
                """, (student_id, analysis_date, analysis_date))
                after_count = cursor.fetchone()[0]
                
                # Mark as effective if improvement
                effective = 1 if after_count > before_count else 0
                
                cursor.execute("""
                    UPDATE agent_analysis_log 
                    SET intervention_effective = ?
                    WHERE id = ?
                """, (effective, log_id))
            
            conn.commit()
            print(f"✅ Evaluated {len(interventions)} interventions")
            return len(interventions)
            
    except Exception as e:
        print(f"❌ Error: {e}")
        return 0

In [31]:
def get_student_analysis(student_id):
    """Get recent AI analysis results"""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("""
        SELECT 
            student_name, 
            alert_level, 
            reason, 
            recommendation,
            analysis_date,
            intervention_effective
        FROM agent_analysis_log 
        WHERE student_id = ?
        ORDER BY created_at DESC 
        """, (student_id,))
        return cursor.fetchall()

In [32]:
get_student_analysis(59)[:1]

[('Larry Bailey',
  'medium',
  '1 absence, no valid circumstances',
  'Contact parents',
  '2025-11-27',
  None)]

In [33]:

def get_session_attendance(date=None):
    """Show attendance per class period """
    if date is None:
        date = datetime.today().strftime('%Y-%m-%d')
    
    with get_connection() as conn:
        cursor = conn.cursor()
        
        cursor.execute("""
        SELECT 
            cs.session_number,
            cs.start_time,
            cs.end_time,
            COUNT(DISTINCT a.student_id) as students_present,
            SUM(CASE WHEN a.attendance_status = 'late' THEN 1 ELSE 0 END) as late_count,
            SUM(CASE WHEN a.attendance_status = 'absent' THEN 1 ELSE 0 END) as absent_count,
            a.attendance_score
        FROM class_schedule cs
        LEFT JOIN attendance_sessions a ON cs.session_number = a.session_number 
            AND a.session_date = ?
        GROUP BY cs.session_number
        ORDER BY cs.session_number
        """, (date,))
        
        return cursor.fetchall()


def get_teacher_dashboard(date=None):
    """Combine everything already have into one view"""
    if date is None:
        date = datetime.today().strftime('%Y-%m-%d')
    
    return {
        'date': date,
        'today_summary': get_session_attendance(date),  
        'weekly_trends': get_weekly_attendance_stats(date), 
        'high_priority': get_high_priority_students(),    
    }

In [34]:
get_session_attendance(59)

[(1, '07:20:00', '08:05:00', 0, 0, 0, None),
 (2, '08:10:00', '08:55:00', 0, 0, 0, None),
 (3, '09:00:00', '09:45:00', 0, 0, 0, None),
 (4, '09:55:00', '10:40:00', 0, 0, 0, None),
 (5, '10:45:00', '11:30:00', 0, 0, 0, None)]

In [35]:
get_teacher_dashboard("2025-11-25")

{'date': '2025-11-25',
 'today_summary': [(1, '07:20:00', '08:05:00', 98, 14, 0, None),
  (2, '08:10:00', '08:55:00', 98, 25, 1, None),
  (3, '09:00:00', '09:45:00', 98, 13, 0, None),
  (4, '09:55:00', '10:40:00', 98, 13, 0, None),
  (5, '10:45:00', '11:30:00', 98, 28, 1, None)],
 'weekly_trends': [('2025-11-21', 84.7, 14.1, 1.2),
  ('2025-11-24', 71.2, 28.0, 0.8),
  ('2025-11-25', 80.6, 19.0, 0.4)],
 'high_priority': [(1, 'John Smith', 10.0, 4),
  (2, 'Emily Johnson', 10.0, 4),
  (3, 'Michael Brown', 10.0, 4),
  (4, 'Sarah Davis', 10.0, 4),
  (5, 'David Wilson', 10.0, 4),
  (6, 'Jennifer Miller', 10.0, 4),
  (7, 'Christopher Moore', 10.0, 4),
  (8, 'Jessica Taylor', 10.0, 4),
  (9, 'Matthew Anderson', 10.0, 4),
  (10, 'Ashley Thomas', 10.0, 4),
  (11, 'James Jackson', 10.0, 4),
  (12, 'Elizabeth White', 10.0, 4),
  (13, 'Daniel Harris', 10.0, 4),
  (14, 'Michelle Martin', 10.0, 4),
  (15, 'Robert Thompson', 10.0, 4),
  (16, 'Laura Garcia', 10.0, 4),
  (17, 'William Martinez', 10.0, 4)

In [36]:
def export_student_report(student_id=None, date=None, format='both'):
    """export student report - generates both CSV and Excel files"""
    try:
        from openpyxl import Workbook
        import csv
        
        if date is None:
            base_filename = "student_attendance_TOTAL_SUMMARY"
        else:
            base_filename = f"student_attendance_{date}"
        
        excel_filename = f"{base_filename}.xlsx"
        csv_filename = f"{base_filename}.csv"
        
        data = []
        headers = ['Student ID', 'Student Name', 'Total Sessions', 
                  'Attended', 'Absent', 'Late', 'Attendance %', 'Avg Score']
        
        if date is not None:
            headers.append('Report Date')
        
        with get_connection() as conn:
            cursor = conn.cursor()
            
            cursor.execute("SELECT id, name FROM students ORDER BY name")
            students = cursor.fetchall()
            
            for sid, name in students:
                if date is None:
                    cursor.execute("""
                        SELECT 
                            COUNT(*) as total,
                            SUM(CASE WHEN attendance_status IN ('on_time', 'late') THEN 1 ELSE 0 END) as attended,
                            SUM(CASE WHEN attendance_status = 'absent' THEN 1 ELSE 0 END) as absent,
                            SUM(CASE WHEN attendance_status = 'late' THEN 1 ELSE 0 END) as late,
                            AVG(attendance_score) as avg_score
                        FROM attendance_sessions
                        WHERE student_id = ?
                    """, (sid,))
                else:
                    cursor.execute("""
                        SELECT 
                            COUNT(*) as total,
                            SUM(CASE WHEN attendance_status IN ('on_time', 'late') THEN 1 ELSE 0 END) as attended,
                            SUM(CASE WHEN attendance_status = 'absent' THEN 1 ELSE 0 END) as absent,
                            SUM(CASE WHEN attendance_status = 'late' THEN 1 ELSE 0 END) as late,
                            AVG(attendance_score) as avg_score
                        FROM attendance_sessions
                        WHERE student_id = ? AND session_date = ?
                    """, (sid, date))
                
                result = cursor.fetchone()
                
                if result and result[0] > 0:
                    total, attended, absent, late, avg_score = result
                    attendance_pct = round((attended / total) * 100, 1) if total > 0 else 0
                    
                    if date is None:
                        row = [sid, name, total, attended, absent, late, 
                               f"{attendance_pct}%", round(avg_score or 0, 2)]
                    else:
                        row = [sid, name, total, attended, absent, late, 
                               f"{attendance_pct}%", round(avg_score or 0, 2), date]
                else:
                    if date is None:
                        row = [sid, name, 0, 0, 0, 0, "0%", 0]
                    else:
                        row = [sid, name, 0, 0, 0, 0, "0%", 0, date]
                
                data.append(row)
        
        if format in ['excel', 'both']:
            wb = Workbook()
            ws = wb.active
            
            if date is None:
                ws.title = "Total Summary Report"
            else:
                ws.title = f"Attendance Report {date}"
            
            ws.append(headers)
            for row in data:
                ws.append(row)
            
            wb.save(excel_filename)
        
   
        if format in ['csv', 'both']:
            with open(csv_filename, 'w', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(headers)
                writer.writerows(data)
        

        if format == 'excel':
            return excel_filename
        elif format == 'csv':
            return csv_filename
        else:  # 'both'
            return {'excel': excel_filename, 'csv': csv_filename}
        
    except ImportError:
        return None

In [37]:
export_student_report()


{'excel': 'student_attendance_TOTAL_SUMMARY.xlsx',
 'csv': 'student_attendance_TOTAL_SUMMARY.csv'}

In [38]:
pd.read_csv("student_attendance_TOTAL_SUMMARY.csv")

,Student ID,Student Name,Total Sessions,Attended,Absent,Late,Attendance %,Avg Score
0,93,Aaron Wallace,27,27,0,3,100.0%,0
1,18,Amanda Robinson,27,25,2,6,92.6%,0
2,88,Andrea Myers,27,27,0,3,100.0%,0
3,48,Anna Flores,27,27,0,7,100.0%,0
4,81,Arthur Foster,27,27,0,6,100.0%,0
...,...,...,...,...,...,...,...,...
95,91,Tyler Graham,27,27,0,2,100.0%,0
96,74,Victoria Fisher,27,27,0,6,100.0%,0
97,77,Walter Gray,27,27,0,6,100.0%,0
98,17,William Martinez,27,27,0,4,100.0%,0


In [39]:
def get_student_attendance_graph_data(student_id, target_date=None, days_before=30):
    """Attendance rate based on present - absent/ total"""
    if target_date is None:
        target_date = datetime.today().strftime('%Y-%m-%d')
    
    with get_connection() as conn:
        cursor = conn.cursor()
        
        cursor.execute("SELECT date(?, ?)", (target_date, f'-{days_before} days'))
        start_date = cursor.fetchone()[0]
        
        cursor.execute("""
        SELECT 
            date(session_date) as day,
            -- Count by status
            COUNT(CASE WHEN attendance_status = 'absent' THEN 1 END) as absent_count,
            COUNT(CASE WHEN attendance_status != 'absent' THEN 1 END) as present_count,
            COUNT(*) as total_sessions
        FROM attendance_sessions 
        WHERE student_id = ? 
          AND session_date >= ? 
          AND session_date <= ?
        GROUP BY date(session_date)
        ORDER BY day
        """, (student_id, start_date, target_date))
        
        # Calculate present rate
        results = []
        for day, absent, present, total in cursor.fetchall():
            if total > 0:
                present_rate = round((present / total) * 100, 1)
            else:
                present_rate = 0
            
            results.append((day, absent, present, total, present_rate))
        
        return results


In [40]:
get_student_attendance_graph_data(15,'2025-11-27')

[('2025-11-21', 1, 4, 5, 80.0),
 ('2025-11-24', 0, 5, 5, 100.0),
 ('2025-11-25', 0, 5, 5, 100.0),
 ('2025-11-26', 0, 5, 5, 100.0),
 ('2025-11-27', 0, 5, 5, 100.0)]

# 1. DAILY INSIGHTS & PRIORITIZATION
"generate_daily_insights" - Generates daily student insights and priority scores
"get_high_priority_students" - Gets students needing immediate attention

# 2. AI ANALYSIS
"analysis_student_problems" - Runs AI analysis on student attendance issues
"evaluate_past_interventions" - Evaluates effectiveness of past interventions

# 3. REPORTS & DASHBOARDS
"get_teacher_dashboard" - Complete daily overview for teachers
"get_session_attendance" - Period-by-period breakdown
"get_weekly_attendance_stats" - Weekly trends and percentages

# 4. SUPPORT FUNCTIONS
"get_intervention_history" - Gets past interventions for a student
"get_recent_analysis" - Gets recent AI recommendations

In [41]:
agent_analytics_tools = [
    "generate_daily_insights",
    "get_high_priority_students", 
    "get_teacher_dashboard",
    "analysis_student_problems", 
    "get_session_attendance"
]

In [42]:
def get_clean_response(result):
    """Extract clean text from agent response"""
    try:
        if isinstance(result, dict) and 'messages' in result:
            last_msg = result['messages'][-1]
            return last_msg.content if hasattr(last_msg, 'content') else str(last_msg)
        elif isinstance(result, dict) and 'output' in result:
            return result['output']
        elif hasattr(result, 'content'):
            return result.content
        else:
            return str(result)
    except Exception as e:
        return f"Error extracting response: {e}"

In [43]:

from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage
llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=os.getenv("GROQ_API_KEY")
)
# # Should use from langchain.agents.middleware import TodoListMiddleware
# agent = create_agent(
#     model=llm_for_agent,
#     tools=[analysis_manager],
#     system_prompt="You are an Attendance Analysis Assistant for a school system."
# )

# # Example usage


# result = agent.invoke(
#     {"messages": [HumanMessage("Generate today insight(but do not show me today insight), only show me the top 5 priority student. Only show me the list of students, don't say anything else or ask")]}
# )
# result = get_clean_response(result)
# print(result)

In [44]:
#pip install sendgrid

In [69]:
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail
def notify_missing_students():
    """
    Send email notifications to parents of missing students using SendGrid.
    Only sends to first 5 students (as requested).
    """
    try:
        # Get today's date
        today = datetime.now().strftime('%Y-%m-%d')
        print(f"Checking attendance for date: {today}")
        
        # Get missing student IDs
        missing_ids = get_missing_students()
        
        if not missing_ids or missing_ids == "No present students":
            print("No missing students to notify today")
            return
        
        # Limit to 5 students
        limited_ids = missing_ids[:5]
        print(f"Notifying parents of {len(limited_ids)} missing student(s)")
        
        # Get student details and parent emails
        with get_connection() as conn:
            cursor = conn.cursor()
            
            # Create placeholders for SQL query
            placeholders = ','.join(['?'] * len(limited_ids))
            
            cursor.execute(f"""
                SELECT name, parent_email 
                FROM students 
                WHERE id IN ({placeholders}) 
                AND parent_email IS NOT NULL 
                AND parent_email != ''
            """, limited_ids)
            
            students_data = cursor.fetchall()
        
        if not students_data:
            print("No parent emails found for missing students")
            return
        
        # Initialize SendGrid
        sendgrid_key = os.getenv("SENDGRID_API_KEY")
        if not sendgrid_key:
            print("SendGrid API key not found in .env file")
            return
            
        sg = SendGridAPIClient(sendgrid_key)
        
        # Send emails
        successful = 0
        for student_name, parent_email in students_data:
            print(f"Sending absence notification for {student_name} to: {parent_email}")
            
            # Email content
            subject = f"Attendance Alert: {student_name} Absent Today"
            html_content = f"""
            <h2>Attendance Notification</h2>
            <p>Dear Parent/Guardian,</p>
            <p>This is to inform you that <strong>{student_name}</strong> was marked <span style="color: #d32f2f; font-weight: bold;">ABSENT</span> from class today ({today}).</p>
            
            <div style="background-color: #f5f5f5; padding: 15px; border-radius: 5px; margin: 15px 0;">
                <p><strong>Absence Details:</strong></p>
                <ul>
                    <li>Date: {today}</li>
                    <li>Student: {student_name}</li>
                    <li>Status: Absent</li>
                </ul>
            </div>
            
            <p>If this absence is due to illness or other valid reasons, please ensure proper documentation is provided to the school.</p>
            
            <p>Regular attendance is crucial for academic success. Please discuss the importance of attendance with your child.</p>
            
            <br>
            <p>Best regards,<br>
            School Attendance System</p>
            """
            
            message = Mail(
                from_email="schoolworkjdoe@gmail.com",
                to_emails=parent_email,
                subject=subject,
                html_content=html_content
            )
            message.reply_to = "schoolworkjdoe@gmail.com"
            
            try:
                response = sg.send(message)
                print(f"  ✅ Email sent! Status: {response.status_code}")
                successful += 1
            except Exception as e:
                print(f"  ❌ Failed to send: {str(e)[:100]}")
        
        print(f"\nResults: {successful}/{len(students_data)} parent notifications sent successfully")
        
    # except Error as e:
    #     print(f"Database error in notify_missing_students: {e}")
    except Exception as e:
        print(f"Error in notify_missing_students: {e}")

In [70]:
notify_missing_students()

Checking attendance for date: 2025-12-13
Notifying parents of 5 missing student(s)
Sending absence notification for Amanda Robinson to: ducduy1982005@gmail.com
  ✅ Email sent! Status: 202
Sending absence notification for Anna Flores to: ducduy1982005@gmail.com
  ✅ Email sent! Status: 202
Sending absence notification for Arthur Foster to: ducduy1982005@gmail.com
  ✅ Email sent! Status: 202
Sending absence notification for Andrea Myers to: ducduy1982005@gmail.com
  ✅ Email sent! Status: 202
Sending absence notification for Aaron Wallace to: ducduy1982005@gmail.com
  ✅ Email sent! Status: 202

Results: 5/5 parent notifications sent successfully


In [49]:
# check_schema.py
import sqlite3

def check_students_table():
    conn = sqlite3.connect('attendance.db')
    cursor = conn.cursor()
    
    print("📋 Checking students table structure...")
    
    # Get all columns in students table
    cursor.execute("PRAGMA table_info(students)")
    columns = cursor.fetchall()
    
    if not columns:
        print("❌ students table doesn't exist!")
    else:
        print("\nCurrent students table columns:")
        for col in columns:
            print(f"  {col[0]}: {col[1]} ({col[2]}) - {'NOT NULL' if col[3] else 'NULLABLE'}")
    
    # Check if parent_email exists
    column_names = [col[1] for col in columns]
    if 'parent_email' not in column_names:
        print("\n❌ parent_email column is MISSING!")
    else:
        print("\n✅ parent_email column exists")
    
    conn.close()

check_students_table()

📋 Checking students table structure...

Current students table columns:
  0: id (INTEGER) - NULLABLE
  1: name (TEXT) - NOT NULL
  2: created_at (TEXT) - NULLABLE
  3: parent_email (TEXT) - NULLABLE

✅ parent_email column exists
